In [1]:
"""Téléchargement des données DVF (Demandes de valeurs foncières) depuis data.gouv.fr.

Le notebook cherche le fichier de l'année 2025 dans la liste des fichiers du jeu de données,
le télécharge, puis le décompresse dans le dossier data/.

Exécuter cette cellule depuis la racine du dépôt pour lancer le téléchargement.
"""

import json
import zipfile
from pathlib import Path
from urllib.request import urlopen, urlretrieve

# Adresse de l'API de data.gouv.fr
API_DVF = "https://www.data.gouv.fr/api/1/datasets/demandes-de-valeurs-foncieres/"

ANNEE = "2025"
DOSSIER_DATA = Path("data")


def trouver_url(annee):
    """Renvoie l'adresse du fichier .zip de l'année demandée."""
    with urlopen(API_DVF) as reponse:
        jeu_de_donnees = json.load(reponse)

    for fichier in jeu_de_donnees["resources"]:
        # On cherche l'année dans le titre, l'adresse ou le format du fichier
        description = f"{fichier.get('title')} {fichier.get('url')} {fichier.get('format')}"
        if annee in description and "zip" in description.lower():
            return fichier["url"]

    # Si rien n'est trouvé, on affiche les fichiers disponibles pour aider à corriger
    titres = [fichier.get("title") for fichier in jeu_de_donnees["resources"]]
    raise ValueError(f"Aucun fichier {annee} trouvé. Fichiers disponibles : {titres}")


def telecharger(url, destination):
    """Télécharge le fichier situé à l'adresse url."""
    print(f"Téléchargement de {url} ...")
    urlretrieve(url, destination)
    print(f"Fichier téléchargé : {destination}")


def decompresser(fichier_zip, dossier):
    """Décompresse le fichier .zip dans le dossier indiqué."""
    with zipfile.ZipFile(fichier_zip) as archive:
        archive.extractall(dossier)
        print("Fichiers extraits :", archive.namelist())


def main():
    DOSSIER_DATA.mkdir(exist_ok=True)

    fichier_txt = DOSSIER_DATA / f"ValeursFoncieres-{ANNEE}.txt"
    if fichier_txt.exists():
        print(f"{fichier_txt} existe déjà : rien à télécharger.")
        return

    fichier_zip = DOSSIER_DATA / f"ValeursFoncieres-{ANNEE}.zip"
    telecharger(trouver_url(ANNEE), fichier_zip)
    decompresser(fichier_zip, DOSSIER_DATA)

    # Le .zip n'est plus utile une fois décompressé
    fichier_zip.unlink()


if __name__ == "__main__":
    main()


Téléchargement de https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20261012-171642/valeursfoncieres-2025.txt.zip ...
Fichier téléchargé : data/ValeursFoncieres-2025.zip
Fichiers extraits : ['ValeursFoncieres-2025.txt']
